STEP 1 : LOAD AND PREPARE DATASET

In [22]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
nltk.download ('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
df = pd.read_csv('spam.csv', encoding='latin-1')

In [5]:
print(df.head())

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
print("Column names:")
print(df.columns)

Column names:
Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')


In [8]:
print("First five rows:")
print(df.head())

First five rows:
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [9]:
print(f"Dataset shape: {df.shape}")

Dataset shape: (5572, 5)


In [10]:
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

In [11]:
print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print("\n" + "="*50 + "\n")

Dataset shape: (5572, 2)

Class distribution:
label
ham     4825
spam     747
Name: count, dtype: int64




In [12]:
print(f"Missing values:\n{df.isnull().sum()}")

Missing values:
label      0
message    0
dtype: int64


STEP 2: TEXT PREPROCESSING

In [13]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

In [14]:
def preprocess_text(text):
    """
    Clean and preprocess the email text
    """
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [ps.stem(word) for word in words if word not in stop_words]
    return ' '.join(words)

In [15]:
print("\nPreprocessing messages...")
df['processed_message'] = df['message'].apply(preprocess_text)

print("\nExample of preprocessing:")
print(f"Original: {df['message'][0]}")
print(f"Processed: {df['processed_message'][0]}")


Preprocessing messages...

Example of preprocessing:
Original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Processed: go jurong point crazi avail bugi n great world la e buffet cine got amor wat


STEP 3: FEATURE EXTRACTION

In [16]:
vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df['processed_message']).toarray()
y = df['label'].map({'ham': 0, 'spam': 1})

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")


Feature matrix shape: (5572, 3000)
Target variable shape: (5572,)


STEP 4: SPLIT DATA

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")


Training set size: 4457
Testing set size: 1115


STEP 5: TRAIN MODEL

In [18]:
print("\nTraining the model...")
model = MultinomialNB()
model.fit(X_train, y_train)
print("Model training complete!")


Training the model...
Model training complete!


In [30]:
# Save model
with open('spam_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save vectorizer
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


STEP 6: EVALUATE MODEL

In [19]:
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n{'='*50}")
print(f"MODEL PERFORMANCE")
print(f"{'='*50}")
print(f"\nAccuracy: {accuracy * 100:.2f}%")

# Fix: Import the missing function
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

# Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print("\n[True Negatives  |  False Positives]")
print("[False Negatives |  True Positives ]")

# Additional metrics
tn, fp, fn, tp = cm.ravel()
print(f"\nDetailed Metrics:")
print(f"True Negatives  (Ham correctly identified)  : {tn}")
print(f"False Positives (Ham wrongly marked as Spam): {fp}")
print(f"False Negatives (Spam missed by model)      : {fn}")
print(f"True Positives  (Spam correctly identified) : {tp}")

precision = tp / (tp + fp)
recall    = tp / (tp + fn)
f1        = 2 * (precision * recall) / (precision + recall)

print(f"\nPrecision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1 Score  : {f1 * 100:.2f}%")


MODEL PERFORMANCE

Accuracy: 97.31%

Classification Report:
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       0.99      0.81      0.89       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115


Confusion Matrix:
[[965   1]
 [ 29 120]]

[True Negatives  |  False Positives]
[False Negatives |  True Positives ]

Detailed Metrics:
True Negatives  (Ham correctly identified)  : 965
False Positives (Ham wrongly marked as Spam): 1
False Negatives (Spam missed by model)      : 29
True Positives  (Spam correctly identified) : 120

Precision : 99.17%
Recall    : 80.54%
F1 Score  : 88.89%


STEP 7: TEST WITH NEW EMAILS

In [20]:
def predict_spam(message):
    """
    Predict if a new message is spam or ham
    """
    # Preprocess the message
    processed = preprocess_text(message)

    # Transform to feature vector
    features = vectorizer.transform([processed]).toarray()

    # Predict
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0]

    result = "SPAM" if prediction == 1 else "HAM"
    confidence = probability[prediction] * 100

    return result, confidence

# Test with sample emails
print(f"\n{'='*50}")
print("TESTING WITH SAMPLE EMAILS")
print(f"{'='*50}\n")

test_messages = [
    "Congratulations! You've won a $1000 gift card. Click here to claim now!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account will be closed. Verify now!",
    "Can you send me the project report by tonight?"
]

for msg in test_messages:
    result, confidence = predict_spam(msg)
    print(f"Message: {msg[:60]}...")
    print(f"Prediction: {result} (Confidence: {confidence:.2f}%)\n")


TESTING WITH SAMPLE EMAILS

Message: Congratulations! You've won a $1000 gift card. Click here to...
Prediction: SPAM (Confidence: 76.88%)

Message: Hey, are we still meeting for lunch tomorrow?...
Prediction: HAM (Confidence: 99.75%)

Message: URGENT: Your account will be closed. Verify now!...
Prediction: SPAM (Confidence: 55.27%)

Message: Can you send me the project report by tonight?...
Prediction: HAM (Confidence: 96.64%)



In [31]:
import pickle

# Load model
with open('spam_model.pkl', 'rb') as f:
    model = pickle.load(f)

# Load vectorizer
with open('vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

In [25]:
def predict_spam(message):
    processed = preprocess_text(message)
    features = vectorizer.transform([processed]).toarray()
    prediction = model.predict(features)[0]

    return "SPAM" if prediction == 1 else "HAM"